# Task 2 — Open-set test
Powtórzenie Task 1 + dodanie ≥100 obrazów osób spoza bazy. Analiza wpływu na FPR.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np, cv2
import matplotlib.pyplot as plt
from pathlib import Path
from src.model import get_insightface_model, get_embedding, cosine_similarity
from src.database import EmbeddingDB
from src.metrics import compute_far_frr, compute_roc, plot_roc
from src.utils import list_images

app = get_insightface_model('buffalo_l', ctx_id=0)
db = EmbeddingDB.from_file()

In [ ]:
# Załaduj obrazy osób spoza bazy (unenrolled) z data/test/
TEST_DIR = Path('../data/test')
enrolled_users = set(db.get_all_users())
scores, labels = [], []

# genuine pairs (enrolled users)
for user_dir in sorted(TEST_DIR.iterdir()):
    if not user_dir.is_dir() or user_dir.name not in enrolled_users: continue
    for img_path in list_images(user_dir):
        img = cv2.imread(str(img_path))
        if img is None: continue
        emb = get_embedding(app, img)
        if emb is None: continue
        refs = db.get_user_embeddings(user_dir.name)
        score = max(cosine_similarity(emb, r) for r in refs)
        scores.append(score); labels.append(1)

# impostor pairs (unenrolled users — open set)
unenrolled_count = 0
for user_dir in sorted(TEST_DIR.iterdir()):
    if not user_dir.is_dir() or user_dir.name in enrolled_users: continue
    if unenrolled_count >= 100: break
    for img_path in list_images(user_dir):
        img = cv2.imread(str(img_path))
        if img is None: continue
        emb = get_embedding(app, img)
        if emb is None: continue
        # score = max similarity against any enrolled user
        all_refs = [r for embs in db.get_all().values() for r in embs]
        score = max(cosine_similarity(emb, r) for r in all_refs)
        scores.append(score); labels.append(0)
        unenrolled_count += 1

scores = np.array(scores); labels = np.array(labels)
print(f'Genuine: {labels.sum()}, Impostor: {(labels==0).sum()}')

In [ ]:
THRESHOLD = 0.4
far, frr, acc = compute_far_frr(scores, labels, THRESHOLD)
print(f'FAR: {far:.4f} | FRR: {frr:.4f} | Acc: {acc:.4f}')

fpr, tpr, _ = compute_roc(scores, labels)
fig, ax = plt.subplots()
plot_roc(fpr, tpr, ax=ax, label='Task 2 Open-set')
plt.savefig('../results/task2/roc.png', dpi=150)
plt.show()